# Session 05 · Clustering Techniques for Customer Segmentation

Run each cell with **Shift+Enter**. Original material by Aseef Ahmed.

In [1]:
# Setup: run from the code/ folder so local modules and data files are found
import os
if os.path.isdir('code'):
    os.chdir('code')

## Example 1 - simulate a transactions log

In [2]:
import numpy as np
from sklearn.cluster import AgglomerativeClustering, KMeans
from sklearn.metrics import adjusted_rand_score, silhouette_score
from sklearn.mixture import GaussianMixture
from shop_data import feature_matrix, make_transactions, rfm_table

tx = make_transactions(n_customers=600, seed=5)
print(tx.head()); print("rows:", len(tx))

  customer  order_id       date  amount  discounted  persona
0    C0000         1 2026-07-11    2192        True  at_risk
1    C0000         2 2025-08-16    3346       False  at_risk
2    C0000         3 2026-02-18    5689       False  at_risk
3    C0001         4 2026-07-10    2925        True  at_risk
4    C0001         5 2025-11-28    4181       False  at_risk
rows: 6072


## Example 2 - build RFM features

In [3]:
rfm = rfm_table(tx)
print(rfm.describe().round(1))

           F         M   disc       R
count  600.0     600.0  600.0   600.0
mean    10.1   48271.5    0.3    76.2
std      9.3   66083.3    0.3   128.8
min      1.0    1315.0    0.0     0.0
25%      4.8   12721.8    0.1     9.0
50%      7.0   20961.5    0.2    30.0
75%     10.0   37691.2    0.5    83.2
max     45.0  326914.0    1.0  1262.0


## Example 3 - k-means with silhouette-based k

In [4]:
X = feature_matrix(rfm)
sil = {k: round(float(silhouette_score(X, KMeans(k, n_init=10, random_state=0).fit_predict(X))), 3) for k in range(2, 8)}
best_k = max(sil, key=sil.get)
print("silhouette:", sil, "best k:", best_k)
km_labels = KMeans(best_k, n_init=10, random_state=0).fit_predict(X)

silhouette: {2: 0.334, 3: 0.424, 4: 0.483, 5: 0.468, 6: 0.442, 7: 0.437} best k: 4


## Example 4 - Gaussian Mixture: BIC and soft membership

In [5]:
bic = {k: round(float(GaussianMixture(k, random_state=0).fit(X).bic(X))) for k in range(2, 8)}
print("BIC:", bic, "lowest at k =", min(bic, key=bic.get))
gmm = GaussianMixture(best_k, random_state=0).fit(X)
proba = gmm.predict_proba(X)
print("first 3 customers' membership:", proba[:3].round(2))

BIC: {2: 3844, 3: 3454, 4: 3246, 5: 3206, 6: 2142, 7: 2156} lowest at k = 6
first 3 customers' membership: [[0.09 0.   0.91 0.  ]
 [0.   0.26 0.74 0.  ]
 [0.   1.   0.   0.  ]]


## Example 5 - agreement between methods and stability

In [6]:
agg_labels = AgglomerativeClustering(best_k, linkage="ward").fit_predict(X)
km_seed1 = KMeans(best_k, n_init=10, random_state=1).fit_predict(X)
print("k-means vs agglomerative ARI:", round(adjusted_rand_score(km_labels, agg_labels), 3))
print("k-means seed 0 vs seed 1 ARI:", round(adjusted_rand_score(km_labels, km_seed1), 3))
print("k-means vs hidden persona ARI:", round(adjusted_rand_score(rfm.persona, km_labels), 3))

k-means vs agglomerative ARI: 0.86
k-means seed 0 vs seed 1 ARI: 1.0
k-means vs hidden persona ARI: 0.68


## Example 6 - profile segments

In [7]:
rfm["segment"] = km_labels
profile = rfm.groupby("segment")[["R", "F", "M", "disc"]].median()
profile["size"] = rfm.segment.value_counts().sort_index()
print(profile.round(2))

             R     F         M  disc  size
segment                                   
0         36.0   7.0   27309.5  0.17   294
1         36.5   6.0   12879.5  0.80   162
2        378.0   4.0   11640.5  0.17    48
3          6.0  30.0  194329.5  0.15    96
